In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd() / "..").resolve()))

from _infra.nbtools import run, mlir_opt_path, tools, artifacts_dir

ART = artifacts_dir()
SRC = Path("../assets/ir/vecadd_linalg.mlir").resolve()

def save_step(name: str, txt: str) -> Path:
    p = ART / f"{name}.mlir"
    p.write_text(txt)
    print("Wrote:", p)
    return p

In [ ]:
from _infra.nbtools import run, mlir_opt_path

pipeline = [
    "-linalg-generalize-named-ops",
    "-one-shot-bufferize=bufferize-function-boundaries",
    "-bufferization-lower-deallocations",
    "-canonicalize", "-cse",
]

txt_buf = run([mlir_opt_path(), *pipeline, str(SRC)])
p2 = save_step("02_bufferized", txt_buf)
print("\nUsed pipeline:\n", " ".join(pipeline))
print("\nPreview:\n", txt_buf[:800])

In [ ]:
pipeline = [
    "-convert-linalg-to-affine-loops",
    "-canonicalize", "-cse",
]
txt_affine = run([mlir_opt_path(), *pipeline, str(p2)])
p3 = save_step("03_affine", txt_affine)
print(txt_affine[:800])

In [ ]:
pipeline = [
    "-lower-affine",
    "-canonicalize", "-cse",
]
txt_scf = run([mlir_opt_path(), *pipeline, str(p3)])
p4 = save_step("04_scf", txt_scf)
print(txt_scf[:800])

In [ ]:
from _infra.nbtools import run, mlir_opt_path

pipeline = [
    "-convert-scf-to-cf",
    "-llvm-request-c-wrappers",
    "-convert-to-llvm",
    "-reconcile-unrealized-casts",
    "-canonicalize",
]

txt_llvm_dialect = run([mlir_opt_path(), *pipeline, str(p4)])
p5 = save_step("05_llvm_dialect", txt_llvm_dialect)
print("\nUsed pipeline:\n", " ".join(pipeline))
print("\nPreview:\n", txt_llvm_dialect[:800])

In [ ]:
mlir_translate = tools().get("mlir-translate")

ll_path = ART / "06_llvm_ir.ll"
ll_txt = run([mlir_translate, "--mlir-to-llvmir", str(p5)])

if isinstance(ll_txt, (bytes, bytearray)):
    ll_txt = ll_txt.decode("utf-8", errors="replace")

ll_path.write_text(ll_txt)
print("Wrote:", ll_path)
print("\n===== LLVM IR (full) =====\n")
print(ll_txt if ll_txt else "(empty)")
